In [25]:
from models.cnn_model import CNN_Model
from utils import (
    EarlyStopping,
    EpochTrainer,
    Face_Detector,
    ResizeKeepRatioPad,
    get_device,
    get_mean_std,
    create_preprocess_config,
    build_cnn_transform,
    build_augmentation_transform,
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
import torchvision.transforms as transforms

# Parameters

In [26]:
batch_size   = 40
epochs       = 20
hidden_size  = 16
kernel_size  = 3
stride       = 1
padding      = 1
pool_size    = 2
input_size   = (100, 100)
patience     = 5
test_size    = 0.3
lr = 0.001

is_face = True

In [ ]:
train_data_path = '../data/face/train/'
model_output_filename = 'cnn_checkpoint'

In [ ]:
criterion = nn.CrossEntropyLoss()

# Get data

In [28]:
stats_steps = []

if is_face:
    stats_steps.append(Face_Detector(input_size=input_size))

stats_steps.extend([
    ResizeKeepRatioPad(input_size, fill=0),
    transforms.ToTensor(),
])

stats_transform = transforms.Compose(stats_steps)

## Option 1 : MNIST dataset

In [29]:
# train_dataset = torchvision.datasets.MNIST(
#     root="../data",
#     train=True,
#     download=True,
#     transform=base_transform,
# )

In [30]:
# test_dataset = torchvision.datasets.MNIST(
#     root="../data",
#     train=False,
#     download=True,
#     transform=base_transform,
# )

## Option 2 : Local dataset

In [31]:
full_dataset = datasets.ImageFolder(    
    root = train_data_path,
    transform=stats_transform,    
)

# Data Processing

## Split data into training and testing

In [32]:
class_names = full_dataset.classes
num_output = len(class_names)


test_records = int(test_size * len(full_dataset) )
train_records = len(full_dataset) - test_records


train_dataset, test_dataset  = random_split(    
    full_dataset, 
    [train_records,test_records],
    generator=torch.Generator().manual_seed(42)
)


## Get number of channels (RGB / Gray) and statistic of dataset

In [33]:
sample_x, _ = train_dataset[0]
channel_size = sample_x.shape[0]

mean, std = get_mean_std(train_dataset)

print(f"class_names: {class_names}")

class_names: ['Ariana-Grande', 'Cristiano-Ronaldo', 'David-Beckham', 'Justin-Bieber', 'Usain-Bolt']


## Normalize data

In [34]:
preprocess_config = create_preprocess_config(
    input_size=input_size,
    mean=mean,
    std=std,
    channel_size=channel_size,
    keep_ratio_pad=True,
    pad_fill=0,
)

In [35]:
augmentation = build_augmentation_transform(
    horizontal_flip=True,
    rotation_degrees=15,
    color_jitter=True,
    random_grayscale_p=0.1,
)
norm_transform = build_cnn_transform(preprocess_config)

In [36]:
base_steps = []

if is_face:
    base_steps.append(Face_Detector(input_size=input_size))


train_steps = base_steps + augmentation.transforms + norm_transform.transforms
val_steps = base_steps + norm_transform.transforms

train_transform = transforms.Compose(train_steps)
val_transform = transforms.Compose(val_steps)

In [37]:
train_dataset.dataset.transform = train_transform
test_dataset.dataset.transform = val_transform

In [38]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Prepare model 

## Prepare config of the model

In [39]:
model_config = {
    "channel_size": channel_size,
    "input_size": tuple(input_size),
    "hidden_size": hidden_size,
    "num_output": num_output,
    "kernel_size": kernel_size,
    "stride": stride,
    "padding": padding,
    "pool_size": pool_size,
}

In [40]:
device = get_device()

In [41]:
model = CNN_Model(
    channel_size=channel_size,
    input_size=input_size,
    hidden_size=hidden_size,
    num_output=num_output,
    kernel_size=kernel_size,
    stride=stride,
    padding=padding,
    pool_size=pool_size,
).to(device)

## Early stopping and checkpoint setup

In [43]:
early_stopping = EarlyStopping(
    patience=patience,
    path=f"../checkpoints/{model_output_filename}.pt",
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "class_names": class_names,
    },
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr = lr)

# Loop epochs and train model

In [44]:
epoch_trainer = EpochTrainer(
    model = model,
    early_stopping = early_stopping,
    device = device,
    optimizer = optimizer,
    criterion = criterion,
    eval_method = 'Accuracy'
)

In [45]:
for epoch in range(epochs):

    avg_train_loss, avg_val_loss, result = epoch_trainer(train_loader , test_loader )

    for key, value in result.items():
        
        print(f"Epoch {epoch + 1:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | {key}: {value:.4f}")
    
    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        
        print("Early stopping triggered! Training stopped.")
        
        break

Epoch   1 | Train Loss: 10.2359 | Val Loss: 6.7477 | Accuracy: 0.3182
Epoch   2 | Train Loss: 4.1700 | Val Loss: 2.5728 | Accuracy: 0.2879
Epoch   3 | Train Loss: 1.2929 | Val Loss: 1.6832 | Accuracy: 0.3939
Epoch   4 | Train Loss: 0.7677 | Val Loss: 1.4730 | Accuracy: 0.4242
Epoch   5 | Train Loss: 0.6443 | Val Loss: 1.3952 | Accuracy: 0.4242
Epoch   6 | Train Loss: 0.4783 | Val Loss: 1.3468 | Accuracy: 0.4697
Epoch   7 | Train Loss: 0.2769 | Val Loss: 1.3836 | Accuracy: 0.4848
Epoch   8 | Train Loss: 0.1535 | Val Loss: 1.5387 | Accuracy: 0.5000
Epoch   9 | Train Loss: 0.0863 | Val Loss: 1.6417 | Accuracy: 0.4848
Epoch  10 | Train Loss: 0.0435 | Val Loss: 1.8402 | Accuracy: 0.5303
Epoch  11 | Train Loss: 0.0245 | Val Loss: 1.9566 | Accuracy: 0.5000
Early stopping triggered! Training stopped.
